# PBL5 - YOLOv8 Damage Detection 2004 v2 (TRAIN FROM SCRATCH)

## Tai sao train lai tu dau?

Dataset **v2 da re-balance** (undersample fold + smart oversample scratch/dirt):
- Fold: 20.1% -> 14.3% (xoa 140 anh pure-fold)
- Scratch: 8.4% -> 10.1% (smart flip + co-occurrence penalty)
- Imbalance ratio: 2.40x -> 1.53x (train)

**Checkpoint cu (`last (8).pt`) da hoc bias phan phoi cu** (fold 20%) -> fine-tune se keo theo bias sai. Tot hon la **train tu yolov8m.pt** (COCO pretrained) de co baseline sach.

## Config v2 - ke thua improvement tu v1_2 analysis:

1. **Train from `yolov8m.pt`** (khong dung checkpoint cu)
2. **`epochs=120, patience=30`** - du dai + early stop
3. **`lr0=0.001`** - standard cho train from scratch (khac fine-tune `1e-4`)
4. **`box=9.0, cls=0.8`** - tang localization, giam cls (classes de phan biet)
5. **`copy_paste=0.3`** - boost minority (scratch/dirt/burn_marks)
6. **`mosaic=1.0, close_mosaic=20`** - aug manh, tat mosaic 20 epoch cuoi
7. **TTA khi validate** - `augment=True` cho +1-2% mAP50

## Uoc luong thoi gian

Kaggle 2xT4 | batch=8 | imgsz=960 | 2326 train imgs:
- ~5-6 phut/epoch -> 120 epochs ≈ **10-12 gio**
- `patience=30` se early stop neu plateau -> thuc te thuong 60-90 epoch

In [ ]:
!pip install -q ultralytics seaborn
import torch, sys, os
from ultralytics import YOLO
print('Python', sys.version.split()[0], '| Torch', torch.__version__, '| CUDA', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} {p.total_memory/1e9:.1f} GB')

## 1. Load dataset + Thong ke

In [ ]:
import os, glob, shutil
from collections import Counter
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

SRC = '/kaggle/input/datasets/quynhunhbo/pb5-train-20-04-test-rasp/yolo_dataset'
DST = '/kaggle/working/yolo_dataset'
CLASS_NAMES = ['material_loss','peel','scratch','fold','writing_marks','dirt','staning','burn_marks']
NC = 8

# Auto-detect neu slug thay doi
if not os.path.isdir(SRC):
    for cand in glob.glob('/kaggle/input/**/images/train', recursive=True):
        SRC = os.path.dirname(os.path.dirname(cand))
        print(f'Auto-detected dataset at: {SRC}')
        break

if not os.path.isdir(f'{DST}/images/train'):
    shutil.copytree(SRC, DST)
    print(f'Copied from {SRC}')
else:
    print(f'Dataset already at {DST}')

# Loai anh hong
removed = 0
for p in glob.glob(f'{DST}/images/**/*', recursive=True):
    if not p.lower().endswith(('.jpg','.jpeg','.png')): continue
    try:
        Image.open(p).verify()
    except Exception:
        lp = os.path.splitext(p.replace('/images/','/labels/'))[0] + '.txt'
        os.remove(p)
        if os.path.exists(lp): os.remove(lp)
        removed += 1
if removed: print(f'Removed {removed} corrupted images')

print('\n' + '='*65)
for split in ('train','val','test'):
    img_dir = f'{DST}/images/{split}'
    lbl_dir = f'{DST}/labels/{split}'
    n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    cnt = Counter()
    for lf in os.listdir(lbl_dir):
        if not lf.endswith('.txt'): continue
        with open(f'{lbl_dir}/{lf}') as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) >= 5: cnt[int(parts[0])] += 1
    tot = sum(cnt.values())
    print(f'\n{split.upper()}: {n_img} images, {tot} annotations')
    for c in range(NC):
        pct = cnt[c]/tot*100 if tot else 0
        print(f'  {c} {CLASS_NAMES[c]:<16s} {cnt[c]:5d} ({pct:5.1f}%)')
vals = [cnt[c] for c in range(NC) if cnt[c] > 0]
if vals:
    print(f'\n  {split} imbalance ratio: {max(vals)/min(vals):.2f}x')
print('='*65)

## 2. data.yaml

In [ ]:
DATA_YAML = '/kaggle/working/data.yaml'
yaml_txt = f"""path: {DST}
train: images/train
val: images/val
test: images/test

nc: 8
names:
  0: material_loss
  1: peel
  2: scratch
  3: fold
  4: writing_marks
  5: dirt
  6: staning
  7: burn_marks
"""
with open(DATA_YAML, 'w') as f:
    f.write(yaml_txt)
print(yaml_txt)

## 3. TRAIN FROM SCRATCH (yolov8m.pt COCO pretrained)

**Auto-resume:** neu session bi cut giua chung, chay lai cell nay se tu dong resume tu `last.pt` trong RUN_DIR.

In [ ]:
RUN_NAME = 'train_2004_v2'
RUN_DIR = f'/kaggle/working/runs/{RUN_NAME}'

last_ckpt = f'{RUN_DIR}/weights/last.pt'
if os.path.exists(last_ckpt):
    print(f'Resume from {last_ckpt}')
    model = YOLO(last_ckpt)
    results = model.train(resume=True)
else:
    model = YOLO('yolov8m.pt')
    print('Start from yolov8m.pt (COCO pretrained)')

    results = model.train(
        data=DATA_YAML,
        epochs=120,
        imgsz=960,
        batch=8,
        device=0,
        project='/kaggle/working/runs',
        name=RUN_NAME,
        exist_ok=True,

        # === OPTIMIZER (train from scratch) ===
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        cos_lr=True,
        warmup_epochs=5,
        warmup_momentum=0.8,
        momentum=0.937,

        # === LOSS WEIGHTS (tang box de cai thien mAP50-95) ===
        box=9.0,
        cls=0.8,
        dfl=1.5,

        # === AUGMENTATION (manh cho train from scratch) ===
        hsv_h=0.01, hsv_s=0.3, hsv_v=0.3,
        degrees=10.0, translate=0.1, scale=0.4, shear=2.0,
        perspective=0.0,
        fliplr=0.5, flipud=0.2,
        mosaic=1.0,
        close_mosaic=20,
        mixup=0.05,
        copy_paste=0.3,
        erasing=0.0,

        # === REGULARIZATION ===
        label_smoothing=0.05,
        dropout=0.0,

        # === TRAINING CONTROL ===
        patience=30,
        save=True,
        save_period=15,
        plots=True,
        val=True,
        amp=True,
        seed=42,
        deterministic=True,
        workers=4,
        cache=False,
    )

print('\n===== TRAINING DONE =====')

## 4. VALIDATE (val set) - TTA

In [ ]:
import numpy as np

BEST_PT = f'/kaggle/working/runs/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_PT)

val_metrics = model.val(
    data=DATA_YAML, split='val',
    imgsz=960, batch=8, device=0,
    conf=0.001, iou=0.6,
    augment=True,  # TTA - +1-2% mAP50
    plots=True, save_json=True,
    project='/kaggle/working/runs', name='val_2004_v2', exist_ok=True,
)

print('\n' + '='*80)
print('VALIDATION RESULTS (val set) - TTA enabled')
print('='*80)
print(f'mAP50:    {val_metrics.box.map50:.4f}')
print(f'mAP50-95: {val_metrics.box.map:.4f}')
print(f'\n{"":>2s} {"Class":<16s} {"AP50":>8s} {"AP50-95":>8s} {"Prec":>8s} {"Recall":>8s} {"F1":>8s}  Status')
print('-'*80)

for i in range(NC):
    ap50 = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0
    ap = val_metrics.box.ap[i] if i < len(val_metrics.box.ap) else 0
    p = val_metrics.box.p[i] if i < len(val_metrics.box.p) else 0
    r = val_metrics.box.r[i] if i < len(val_metrics.box.r) else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    st = 'OK' if ap50 > 0.5 else ('WARN' if ap50 > 0.3 else 'WEAK')
    print(f'{i:>2d} {CLASS_NAMES[i]:<16s} {ap50:8.4f} {ap:8.4f} {p:8.4f} {r:8.4f} {f1:8.4f}  {st}')

avg_p = np.mean([val_metrics.box.p[i] for i in range(NC)])
avg_r = np.mean([val_metrics.box.r[i] for i in range(NC)])
avg_f1 = 2*avg_p*avg_r/(avg_p+avg_r) if (avg_p+avg_r) > 0 else 0
print(f'\n{"":>2s} {"AVERAGE":<16s} {val_metrics.box.map50:8.4f} {val_metrics.box.map:8.4f} {avg_p:8.4f} {avg_r:8.4f} {avg_f1:8.4f}')

## 5. TEST (test set) - Danh gia cuoi cung

In [ ]:
print('='*80)
print('TEST SET EVALUATION - Du lieu chua tung thay (TTA enabled)')
print('='*80)

test_metrics = model.val(
    data=DATA_YAML, split='test',
    imgsz=960, batch=8, device=0,
    conf=0.001, iou=0.6,
    augment=True,
    plots=True, save_json=True,
    project='/kaggle/working/runs', name='test_2004_v2', exist_ok=True,
)

print(f'\nmAP50:    {test_metrics.box.map50:.4f}')
print(f'mAP50-95: {test_metrics.box.map:.4f}')
print(f'\n{"":>2s} {"Class":<16s} {"AP50":>8s} {"AP50-95":>8s} {"Prec":>8s} {"Recall":>8s} {"F1":>8s}')
print('-'*80)

for i in range(NC):
    ap50 = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0
    ap = test_metrics.box.ap[i] if i < len(test_metrics.box.ap) else 0
    p = test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0
    r = test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f'{i:>2d} {CLASS_NAMES[i]:<16s} {ap50:8.4f} {ap:8.4f} {p:8.4f} {r:8.4f} {f1:8.4f}')

avg_p = np.mean([test_metrics.box.p[i] for i in range(NC)])
avg_r = np.mean([test_metrics.box.r[i] for i in range(NC)])
avg_f1 = 2*avg_p*avg_r/(avg_p+avg_r) if (avg_p+avg_r) > 0 else 0
print(f'\n{"":>2s} {"AVERAGE":<16s} {test_metrics.box.map50:8.4f} {test_metrics.box.map:8.4f} {avg_p:8.4f} {avg_r:8.4f} {avg_f1:8.4f}')

print(f'\n{"="*80}')
print('SO SANH VAL vs TEST (generalization check):')
print(f'  Val  mAP50: {val_metrics.box.map50:.4f}  |  mAP50-95: {val_metrics.box.map:.4f}')
print(f'  Test mAP50: {test_metrics.box.map50:.4f}  |  mAP50-95: {test_metrics.box.map:.4f}')
gap50 = val_metrics.box.map50 - test_metrics.box.map50
gap = val_metrics.box.map - test_metrics.box.map
print(f'  Gap mAP50:    {gap50:+.4f}')
print(f'  Gap mAP50-95: {gap:+.4f}')
if abs(gap50) < 0.03:
    print('  => Generalize TOT')
elif gap50 > 0.05:
    print('  => OVERFIT. Giam epochs hoac tang augmentation')
elif gap50 > 0.03:
    print('  => Dau hieu overfit nhe')
else:
    print('  => Test tot hon val (ngau nhien do split)')
print('='*80)

## 6. Training Curves

In [ ]:
import csv
import matplotlib.pyplot as plt

results_csv = f'/kaggle/working/runs/{RUN_NAME}/results.csv'
if os.path.exists(results_csv):
    data = {}
    with open(results_csv) as f:
        reader = csv.DictReader(f)
        for row in reader:
            for key, val in row.items():
                k = key.strip()
                try: data.setdefault(k, []).append(float(val))
                except: pass

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Training Curves - {RUN_NAME}', fontsize=14, fontweight='bold')

    for idx, (tk, vk, title) in enumerate([
        ('train/box_loss','val/box_loss','Box Loss'),
        ('train/cls_loss','val/cls_loss','Cls Loss'),
        ('train/dfl_loss','val/dfl_loss','DFL Loss'),
    ]):
        ax = axes[0][idx]
        if tk in data: ax.plot(data[tk], label='Train', color='#3498db', alpha=0.8)
        if vk in data: ax.plot(data[vk], label='Val', color='#e74c3c', alpha=0.8)
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

    for idx, (key, title, color) in enumerate([
        ('metrics/precision(B)','Precision','#2ecc71'),
        ('metrics/recall(B)','Recall','#e67e22'),
        ('metrics/mAP50(B)','mAP50','#9b59b6'),
    ]):
        ax = axes[1][idx]
        if key in data:
            vals = data[key]
            ax.plot(vals, color=color, alpha=0.8, linewidth=1.5)
            best_ep = int(np.argmax(vals))
            ax.plot(best_ep, vals[best_ep], 'r*', markersize=12)
            ax.set_title(f'{title} | Best: {vals[best_ep]:.4f} @ ep {best_ep}')
        ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3); ax.set_ylim(0,1)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'results.csv not found at {results_csv}')

## 7. Confusion Matrix + PR/F1 Curves

In [ ]:
from IPython.display import Image as IPImage, display

search_dirs = [
    '/kaggle/working/runs/test_2004_v2',
    '/kaggle/working/runs/val_2004_v2',
    f'/kaggle/working/runs/{RUN_NAME}',
]

for fname, desc in [
    ('F1_curve.png', 'F1 CURVE'),
    ('P_curve.png',  'PRECISION CURVE'),
    ('R_curve.png',  'RECALL CURVE'),
    ('PR_curve.png', 'PR CURVE'),
]:
    for d in search_dirs:
        p = f'{d}/{fname}'
        if os.path.exists(p):
            tag = 'TEST' if 'test' in d else ('VAL' if 'val' in d else 'TRAIN')
            print(f'\n--- {desc} ({tag}) ---')
            display(IPImage(p, width=800))
            break

for fname in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    for d in search_dirs:
        p = f'{d}/{fname}'
        if os.path.exists(p):
            tag = 'TEST' if 'test' in d else ('VAL' if 'val' in d else 'TRAIN')
            norm = '(NORMALIZED)' if 'normalized' in fname else ''
            print(f'\n--- CONFUSION MATRIX {tag} {norm} ---')
            display(IPImage(p, width=700))
            break

## 8. Per-class Metrics (Val vs Test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

val_ap = [val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0 for i in range(NC)]
test_ap = [test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0 for i in range(NC)]
x = np.arange(NC); w = 0.4

ax = axes[0]
ax.barh(x - w/2, val_ap, w, label=f'Val (mAP50={val_metrics.box.map50:.3f})', color='#3498db', alpha=0.8)
ax.barh(x + w/2, test_ap, w, label=f'Test (mAP50={test_metrics.box.map50:.3f})', color='#e74c3c', alpha=0.8)
ax.set_yticks(x); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('AP50'); ax.set_title('AP50: Val vs Test'); ax.legend(); ax.set_xlim(0,1)
ax.invert_yaxis(); ax.grid(True, alpha=0.3, axis='x')

ax = axes[1]
p_vals = [test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0 for i in range(NC)]
r_vals = [test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0 for i in range(NC)]
f1_vals = [2*p*r/(p+r) if (p+r)>0 else 0 for p,r in zip(p_vals, r_vals)]
colors_f1 = ['#e74c3c' if v<0.3 else '#f39c12' if v<0.5 else '#27ae60' for v in f1_vals]
bars = ax.barh(CLASS_NAMES, f1_vals, color=colors_f1)
for bar, val in zip(bars, f1_vals):
    ax.text(max(bar.get_width()+0.02,0.05), bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center')
ax.set_xlabel('F1'); ax.set_title('F1 Score (Test)'); ax.set_xlim(0,1)
ax.axvline(x=np.mean(f1_vals), color='blue', ls='--', alpha=0.5, label=f'Mean: {np.mean(f1_vals):.3f}')
ax.legend(loc='lower right'); ax.invert_yaxis(); ax.grid(True, alpha=0.3, axis='x')

ax = axes[2]
ax.bar(x-w/2, p_vals, w, label='Precision', color='#3498db')
ax.bar(x+w/2, r_vals, w, label='Recall', color='#e67e22')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_title('Precision vs Recall (Test)'); ax.legend(); ax.set_ylim(0,1); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/kaggle/working/per_class_metrics_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visual Predictions (Test set)

In [ ]:
import random

test_dir = f'{DST}/images/test'
test_imgs = sorted(glob.glob(f'{test_dir}/*'))
print(f'Test dir: {test_dir} ({len(test_imgs)} images)')

random.seed(42)
sample = random.sample(test_imgs, min(16, len(test_imgs)))

model.predict(
    sample, save=True, imgsz=960, conf=0.25, iou=0.5, max_det=50,
    project='/kaggle/working/runs', name='predict_test_v2', exist_ok=True,
)
for img_path in sorted(glob.glob('/kaggle/working/runs/predict_test_v2/*'))[:12]:
    if img_path.lower().endswith(('.jpg','.jpeg','.png')):
        display(IPImage(img_path, width=600))
        print()

## 10. Error Analysis

In [ ]:
print('='*80)
print('ERROR ANALYSIS (Test set)')
print('='*80)
print(f'\nOverall: mAP50={test_metrics.box.map50:.4f}, mAP50-95={test_metrics.box.map:.4f}')

n_ok = n_warn = n_weak = 0
for i in range(NC):
    ap50 = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0
    if ap50 >= 0.5: n_ok += 1
    elif ap50 >= 0.3: n_warn += 1
    else: n_weak += 1
print(f'  OK (>0.5): {n_ok}/{NC} | WARN (0.3-0.5): {n_warn}/{NC} | WEAK (<0.3): {n_weak}/{NC}')

print(f'\nCac class can cai thien:')
any_weak = False
for i in range(NC):
    ap50 = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0
    if ap50 >= 0.5: continue
    any_weak = True
    p_i = test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0
    r_i = test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0
    print(f'  [{i}] {CLASS_NAMES[i]}: AP50={ap50:.4f} P={p_i:.4f} R={r_i:.4f}')
    if r_i < 0.4:
        print(f'      -> Recall thap: bo qua nhieu, can them data / giam conf threshold')
    if p_i < 0.4:
        print(f'      -> Precision thap: false positive, can clean labels')
if not any_weak:
    print('  Tat ca class dat AP50 > 0.5!')

gap = val_metrics.box.map50 - test_metrics.box.map50
print(f'\nGeneralization: val-test gap = {gap:+.4f}')
if abs(gap) < 0.03:
    print('  -> TOT')
elif gap > 0.05:
    print('  -> OVERFIT')
print('='*80)

## 11. Save outputs

In [ ]:
import json

RUN_DIR = f'/kaggle/working/runs/{RUN_NAME}'

for name in ['best.pt', 'last.pt']:
    src = f'{RUN_DIR}/weights/{name}'
    if os.path.exists(src):
        shutil.copy2(src, f'/kaggle/working/{name}')
        print(f'{name}: {os.path.getsize(src)/1e6:.1f} MB')

save_files = [
    'results.png','results.csv',
    'confusion_matrix.png','confusion_matrix_normalized.png',
    'F1_curve.png','P_curve.png','R_curve.png','PR_curve.png',
    'labels.jpg','labels_correlogram.jpg',
]
saved = []
all_dirs = [RUN_DIR, '/kaggle/working/runs/val_2004_v2', '/kaggle/working/runs/test_2004_v2']
for f in save_files:
    for d in all_dirs:
        s = f'{d}/{f}'
        if os.path.exists(s):
            shutil.copy2(s, f'/kaggle/working/{f}')
            saved.append(f); break

for f in ['training_curves_v2.png','per_class_metrics_v2.png']:
    if os.path.exists(f'/kaggle/working/{f}'): saved.append(f)

report = {
    'run': RUN_NAME,
    'dataset': 'pbl5-dataset-2004-8class-v2 (re-balanced: undersample + smart oversample)',
    'model': 'yolov8m from scratch (COCO pretrained)',
    'imgsz': 960,
    'config': {
        'lr0': 0.001, 'epochs': 120, 'patience': 30,
        'box': 9.0, 'cls': 0.8, 'dfl': 1.5,
        'copy_paste': 0.3, 'mosaic': 1.0, 'close_mosaic': 20,
    },
    'val': {
        'mAP50': round(float(val_metrics.box.map50), 4),
        'mAP50_95': round(float(val_metrics.box.map), 4),
    },
    'test': {
        'mAP50': round(float(test_metrics.box.map50), 4),
        'mAP50_95': round(float(test_metrics.box.map), 4),
    },
    'per_class_test_ap50': {
        CLASS_NAMES[i]: round(float(test_metrics.box.ap50[i]), 4)
        for i in range(NC) if i < len(test_metrics.box.ap50)
    }
}
with open('/kaggle/working/test_report_v2.json', 'w') as f:
    json.dump(report, f, indent=2)
saved.append('test_report_v2.json')

print(f'\nSaved {len(saved)} files to /kaggle/working/')
print(json.dumps(report, indent=2))

## 12. RESUME (neu session bi ngat)

Chay cell nay neu Kaggle cut session giua chung. Tu dong tim last.pt va resume.

In [ ]:
from ultralytics import YOLO
import os, glob

candidates = [
    f'/kaggle/working/runs/{RUN_NAME}/weights/last.pt',
    '/kaggle/working/last.pt',
] + sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True))

for c in candidates:
    if os.path.exists(c):
        print(f'Resuming from: {c}')
        YOLO(c).train(resume=True)
        print('Done!')
        break
else:
    print('No checkpoint found. Run section 3 to start training.')